# Imports

In [ ]:
import logging, warnings; logging.getLogger().setLevel(logging.ERROR);
warnings.filterwarnings("ignore")

import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import re
from pathlib import Path 

import warnings, scipy.sparse as sp, matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.pyplot import rc_context
import matplotlib.font_manager
import matplotlib.lines as lines


pd.set_option('display.max_rows', 200)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rc('font', size=12)

sc.settings.n_jobs=-1
sc.set_figure_params(dpi=80, dpi_save=300, color_map='Spectral_r', vector_friendly=True, transparent=True)
sc.settings.figdir = '../../1_outputs/0_figures'
sc.settings.verbosity = 1 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

%matplotlib inline 
%config InlineBackend.figure_format = 'retina'

In [ ]:
def observe_variance(anndata_object):
    fig = plt.figure(figsize=(10,5))
    ax1 = fig.add_subplot(121)
    ax2 = fig.add_subplot(122)
    # variance per principal component
    x = range(len(anndata_object.uns['pca']['variance_ratio']))
    y = anndata_object.uns['pca']['variance_ratio']
    ax1.scatter(x,y,s=4)
    ax1.set_xlabel('PC')
    ax1.set_ylabel('Fraction of variance explained\n')
    ax1.set_title('Fraction of variance explained per PC\n')
    # cumulative variance explained
    cml_var_explained = np.cumsum(anndata_object.uns['pca']['variance_ratio'])
    x = range(len(anndata_object.uns['pca']['variance_ratio']))
    y = cml_var_explained
    ax2.scatter(x,y,s=4)
    ax2.set_xlabel('PC')
    ax2.set_ylabel('Cumulative fraction of variance\nexplained')
    ax2.set_title('Cumulative fraction of variance\nexplained by PCs')
    fig.tight_layout()
    plot = plt.show
    return(plot)

In [ ]:
# preset color palettes and color maps
user_defined_palette =  [ '#F6222E', '#16FF32', '#3283FE', '#FEAF16', '#BDCDFF', '#3B00FB', '#1CFFCE', '#C075A6', '#F8A19F', '#B5EFB5', '#FBE426', '#C4451C', 
                          '#2ED9FF', '#c1c119', '#8b0000', '#FE00FA', '#1CBE4F', '#1C8356', '#0e452b', '#AA0DFE', '#B5EFB5', '#325A9B', '#90AD1C']

user_defined_cmap_markers = LinearSegmentedColormap.from_list('mycmap', ["#E6E6FF", "#CCCCFF", "#B2B2FF", "#9999FF",  "#6666FF",   "#3333FF", "#0000FF"])
user_defined_cmap_degs = LinearSegmentedColormap.from_list('mycmap', ["#0000FF", "#3333FF", "#6666FF", "#9999FF", "#B2B2FF", "#CCCCFF", "#E6E6FF", "#E6FFE6", "#CCFFCC", "#B2FFB2", "#99FF99", "#66FF66", "#33FF33", "#00FF00"])

In [ ]:
pwd

In [ ]:
file_outputs = '../1_outputs/2_deg/0_dn/'
h5ad = '../../1_outputs/1_h5ad/'

In [ ]:
adata = sc.read_h5ad(h5ad + '7_nk_ilc.h5ad')
adata

## Rerun HVG, PCA, Harmony, UMAP

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(adata, n_comps=50, svd_solver='arpack', random_state=42, use_highly_variable=False) #change use_highly_variable == True if you ran the block above

In [ ]:
observe_variance(adata)

In [ ]:
sc.tl.pca(adata, n_comps=13, svd_solver='arpack', random_state=42, use_highly_variable=False) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
sce.pp.harmony_integrate(adata, 'sample', max_iter_harmony = 20) # Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, random_state=42, use_rep='X_pca_harmony')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(adata, random_state=42)

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['sample', 'condition'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    #size=5,
    frameon=False,
    add_outline=True,
    sort_order = False
)

# Annotations

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    adata, 
    color=[
        'Ptprc',

        #T cells: 
        'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Foxp3', 'Il2ra', 'Gzmk',

        #NK cells
        'Klrb1c', 'Ncr1', 'Nkg7',
        
        #B Cells:
        'Cd19', 'Cd27',

        #Plasma Cells: 
        'Tnfrsf17',

        #Myeloid/ DCS
        'Itgam', 'Itgax', 'Sirpa', 'Clec9a', 'Xcr1', 'Siglech', 'Arg1', 'Ly6c1',

        'Mki67', 'Il18r1', 'Mki67'
        
        
          ],  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    vmax='p99',
    frameon=False,
    add_outline=True,
    sort_order = False,
)

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    adata, 
    color=[
        'Il1rl1', 'Il7r', 'Zfp683', 'Rora', 'Rorc', 'Eomes', 'Tbx21',
        'Cxcr6', 'Cd226', 'Il1r1', 'Ncr1', 'Cd3e', 'Cd3g', 'Cd3d',
        'Zbtb16', 'Il23r', 'Cd44', 'Ccr6', 'Il22', 'Il17rc',

        'Trac', 'Trdc',

        'Mr1', 'Trav1'
          ],  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    vmax='p99',
    frameon=False,
    add_outline=True,
    sort_order = False,
)

In [ ]:

sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    adata, 
    color=[
       
    # General ILC / progenitor
    "Il7r", "Thy1", "Id2", "Tcf7", "Tox",

    # ILCP
    "Zbtb16", "Pdcd1", "Kit",

    # ILC1
    "Tbx21", "Itga1", "Cxcr6", "Il18r1", "Ifng",

    # NK distinction
    "Eomes", "Ncr1", "Nkg7", "Prf1",

    # ILC2
    "Gata3", "Rora", "Il1rl1", "Il17rb", "Areg", "Il5", "Il13",

    # ILC3
    "Rorc", "Il23r", "Ccr6", "Ahr", "Ltb", "Il22",

    # Adaptive-lineage exclusion
    "Cd3d", "Cd3e", "Trac", "Cd79a",
],  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    vmax='p99',
    frameon=False,
    add_outline=True,
    sort_order = False,
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0
 
for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
                             1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]:
    sc.tl.leiden(adata, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    adata, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 
        'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.tl.rank_genes_groups(adata, 
                        groupby='leiden_0.3',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

#df.to_csv(file_outputs + 'ilc_umbrella_leiden_0.1.csv', index = False)

df.head(25)

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    adata, 
    color=[
        'leiden_0.3', 'Trac', 'Trdc', 'Il1r1', 'Cd4', 'Cd8a', 'Cd8b1', 
        'Ccr9', 'Gzmb','Sell', 'Klrb1', 'Slc4a10', 'Mki67',
        'Tbx21', 'Eomes', 'Il2rb', 'Fasl', 'Itgae', 'Nrp1', 'Rorc',
        'Il18r1', 'Il18rap', 'Itga1', 'Il23r', 'Il1r1', 'Trdc', 'Trgv2', 'Zbtb16',
        'Il1rl1', 'Rora', 'Gata3', 'Il2ra', 'Il4', 'Stat4', 'Il23r', 'H2-K1', 
        'Nfkbia', 'Ccr9', 'Cd28', 'Cd53'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False,
    vmax = 'p99'
   # save='_ilc_umaps.pdf'
)

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    adata, 
    color=[
        'leiden_0.3',
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False,
    vmax = 'p99'
   # save='_ilc_umaps.pdf'
)

In [ ]:
sc.tl.rank_genes_groups(adata, 
                        groupby='leiden_0.3',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

#df.to_csv(file_outputs + 'final_degs.csv', index = False)

df.head(35)

In [ ]:
marker_genes = ['Il23r', 'Il1r1', 'Zbtb16', 'Rora', 'Gata3', 'Il2ra', 'Il4', 'Stat4']

sc.pl.dotplot(adata, 
              marker_genes, 
              groupby='leiden_0.3', 
              use_raw=False, 
              standard_scale='var', dendrogram=True)

In [ ]:
cell_type_groups = {
    'T': ['0', '1', '2', '3', '4', '6', '8'],
    'NK': ['5'], 
    #'ILC:ILCP': ['0', '1', '3', '4', '5', '6', '9', '11', '13', '8', '12'],
    'ILC:ILC3': ['7'], 
    #'Unknown ILC/NK': ['8', '12'], 
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
# cell_type_groups = {
#     'T:CD8 GZMK+': ['10'],
#     'NK': ['2'], 
#     'ILC:ILCP': ['0', '1', '3', '4', '5', '6', '9', '11', '13', '8', '12'],
#     'ILC:ILC1': ['7'], 
#     #'Unknown ILC/NK': ['8', '12'], 
# }

# cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
adata.obs['cell_type_subset'] = adata.obs['leiden_0.3'].map(cluster_to_cell_type) #Add Leiden Cluster

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=[
        'cell_type_subset',
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

## Subset Unknowns

In [ ]:
uk = adata[adata.obs['cell_type_subset'].isin(['T'])].copy()
uk

In [ ]:
sc.pp.highly_variable_genes(uk, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(uk, n_comps=50, svd_solver='arpack', random_state=42, use_highly_variable=False) #change use_highly_variable == True if you ran the block above

In [ ]:
observe_variance(uk)

In [ ]:
sc.tl.pca(uk, n_comps=35, svd_solver='arpack', random_state=42, use_highly_variable=False) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
sce.pp.harmony_integrate(uk, 'sample', max_iter_harmony = 20) # Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
sc.pp.neighbors(uk, n_neighbors=15, random_state=42, use_rep='X_pca_harmony')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(uk, random_state=42)

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    uk, 
    color=['sample', 'condition'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    #size=5,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:

sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    uk, 
    color=[

    'Cd4', 'Cd8a',
       
    # General ILC / progenitor
    "Il7r", "Thy1", "Id2", "Tcf7", "Tox",

    # ILCP
    "Zbtb16", "Pdcd1", "Kit",

    # ILC1
    "Tbx21", "Itga1", "Cxcr6", "Il18r1", "Ifng",

    # NK distinction
    "Eomes", "Ncr1", "Nkg7", "Prf1",

    # ILC2
    "Gata3", "Rora", "Il1rl1", "Il17rb", "Areg", "Il5", "Il13",

    # ILC3
    "Rorc", "Il23r", "Ccr6", "Ahr", "Ltb", "Il22",

    # Adaptive-lineage exclusion
    "Cd3d", "Cd3e", "Trac", "Cd79a",
],  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    vmax='p99',
    frameon=False,
    add_outline=True,
    sort_order = False,
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0
 
for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
                             1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]:
    sc.tl.leiden(uk, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    uk, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 
        'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    uk, 
    color=[
        'leiden_1.6',
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False,
    vmax = 'p99'
   # save='_ilc_umaps.pdf'
)

In [ ]:
cell_type_groups = {
    'T:CD4': ['5', '8', '10', '11', '18'],
    'T:CD8(GZMK+)': ['13'], 
    'ILC:ILCP': ['0', '1', '2', '3', '4', '6', '7', '9', '14', '15'],
    'ILC:ILC1': ['12','17'], 
    'ILC:ILC2': ['16'], 

    #'Unknown ILC/NK': ['8', '12'], 
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
uk.obs['cell_type_subset'] = uk.obs['leiden_1.6'].map(cluster_to_cell_type) #Add Leiden Cluster

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    uk, 
    color=[
        'cell_type_subset',
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

## Map Subsets

In [ ]:
adata.obs['cell_type_subset'].value_counts().sort_index()

In [ ]:
temp = adata[~adata.obs['cell_type_subset'].isin(['T'])].obs['cell_type_subset'].copy()
temp

In [ ]:
with_renamed_subsets = pd.concat([uk.obs['cell_type_subset'], 
                                  temp])

In [ ]:
adata.obs['cell_type_subset'] = ''

In [ ]:
adata.obs['cell_type_subset'][adata.obs.index.isin(with_renamed_subsets.index) == True] = with_renamed_subsets

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(adata, n_comps=75, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
sce.pp.harmony_integrate(adata, ['sample'], max_iter_harmony = 20)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, random_state=42, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(adata, random_state = 42) # min_dist=0.2, spread = 1.3,

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['cell_type_subset'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    size = 10,
    sort_order = False, 
    #vmax='p99'
)

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['cell_type_subset', 

           "Rorc",
                   "Ahr",
                   "Il23r",
                   "Il1r1",
                   "Ccr6"


            ],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    size = 10,
    sort_order = False, 
    #vmax='p99'
)

In [ ]:
sc.tl.rank_genes_groups(adata,
groupby='cell_type_subset', 
method='wilcoxon', 
use_raw=False)

result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150) #'scores', 'logfoldchanges', 'pvals_adj'

# df.to_csv(file_outputs + '0_umbrella_gene_rank.csv', index = False)
df.head(50)


In [ ]:
adata

In [ ]:
adata.write_h5ad(h5ad + '7_nk_ilc.h5ad')